In [ ]:
# DINOv3 está disponível no Transformers a partir da versão 4.56.
# torchao é atualizado para evitar incompatibilidades com versões recentes do PEFT.
%pip install -q -U \
    "torchao>=0.16.0" \
    "transformers>=4.56.0,<6" \
    "peft>=0.17.0" \
    "accelerate>=1.0" \
    "huggingface_hub>=0.34" \
    "scikit-learn>=1.3" \
    "pandas>=2.0" \
    "pillow<12.0" \
    "tqdm" \
    "matplotlib" \
    "opencv-python-headless"


In [ ]:
import sys
import torch
import transformers
import peft

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")


In [ ]:
# Monte o Google Drive.
from google.colab import drive

drive.mount("/content/drive")

# Os modelos DINOv3 podem exigir aceite da licença no Hugging Face.
from huggingface_hub import notebook_login

# notebook_login()


In [ ]:
# ====== CAMINHOS ======
CSV_PATH = "/content/drive/MyDrive/BRSET/labels_brset.csv"
IMAGES_DIR = "/content/drive/MyDrive/BRSET/fundus_photos"
OUTPUT_DIR = (
    "/content/drive/MyDrive/BRSET/retinascan_cbis/"
    "exp_hr_dinov3_large_base_four_fusion_strategies"
)

# ====== TAREFA ======
TASK_NAME = "HR (Normal vs Hypertensive Retinopathy)"
TARGET_COL = "hypertensive_retinopathy"
POS_LABEL_NAME = "Hypertensive Retinopathy"
NEG_LABEL_NAME = "Normal"
IMAGE_EXT = ".jpg"

# ====== BACKBONES E FUSÃO ======
BACKBONES = {
    "large": "facebook/dinov3-vitl16-pretrain-lvd1689m",
    "base": "facebook/dinov3-vitb16-pretrain-lvd1689m",
}
FUSION_MODES = [
    "image_only",
    "visual_patch",
    "learned_tokens",
    "late_fusion",
]
EXPERIMENTS_TO_RUN = [
    ("large", "image_only"),
    ("large", "visual_patch"),
    ("large", "learned_tokens"),
    ("large", "late_fusion"),
    ("base", "image_only"),
    ("base", "visual_patch"),
    ("base", "learned_tokens"),
    ("base", "late_fusion"),
]

# Para teste rápido use [0]. Para o estudo completo use [0, 1, 2].
FOLDS_TO_RUN = [0, 1, 2]
# O diretório foi alterado para não reutilizar resultados do protocolo anterior.
# Depois da primeira execução completa, pode ser alterado para True.
SKIP_COMPLETED = False
RUN_SMOKE_TEST = False

# ====== IMAGEM / PRÉ-PROCESSAMENTO ======
IMAGE_SIZE = 512
USE_ROBUST_CIRCLE_CROP = True
CIRCLE_CROP_PADDING = 1.03
CIRCLE_DETECT_MAX_SIDE = 768
PRE_RESIZE_BEFORE_CROP = True
PRE_RESIZE_MAX_SIDE = 1200
USE_CLAHE = False
CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID_SIZE = (8, 8)
META_PATCH_SIZE = 32
META_PATCH_QUAD = 16
EXCLUDE_INADEQUATE_QUALITY = False
USE_METADATA_IMAGE_PATCH = True  # usado apenas pelas células antigas de preview

# ====== PROTOCOLO DE REFERÊNCIA ======
# Os hiperparâmetros abaixo reproduzem o protocolo do notebook isolado:
# notebook corrigido de retinopatia diabética, alterando apenas a doença-alvo.
PROTOCOL_TAG = "hr_same_protocol_as_corrected_dr_notebook"
RESET_SEED_PER_FOLD_AND_STRATEGY = True

# ====== TREINO ======
BATCH_SIZE_BY_BACKBONE = {"large": 16, "base": 16}
GRAD_ACCUM_STEPS = 4
NUM_WORKERS = 16
EPOCHS = 50
PATIENCE = 7
LORA_LR = 1e-4
HEAD_METADATA_LR = 3e-4
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 1.0

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "up_proj",
    "down_proj",
]

FOCAL_GAMMA = 1.0
CLASS_WEIGHT_BETA = 0.999
USE_OVERSAMPLING = False
USE_AMP = True
USE_GRADIENT_CHECKPOINTING = False

# ====== CV ======
N_FOLDS = 3
VAL_SIZE = 0.15
N_BOOT = 1000
SEED = 42

assert all(b in BACKBONES for b, _ in EXPERIMENTS_TO_RUN)
assert all(m in FUSION_MODES for _, m in EXPERIMENTS_TO_RUN)

print("Grade experimental:")
for i, (backbone_key, fusion_mode) in enumerate(EXPERIMENTS_TO_RUN, 1):
    print(f"{i:02d}. {backbone_key:5s} | {fusion_mode}")


# Conferência automática do protocolo do experimento isolado.
assert BATCH_SIZE_BY_BACKBONE["large"] == 16
assert GRAD_ACCUM_STEPS == 4
assert NUM_WORKERS == 16
assert EPOCHS == 50
assert PATIENCE == 7
assert LORA_LR == 1e-4
assert HEAD_METADATA_LR == 3e-4
assert WEIGHT_DECAY == 1e-4
assert LORA_R == 16
assert LORA_ALPHA == 32
assert LORA_DROPOUT == 0.05
assert LORA_TARGET_MODULES == [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "up_proj",
    "down_proj",
]
print("\n✓ Protocolo de treinamento mantido; apenas a doença-alvo foi alterada.")


In [ ]:
import gc
import os
import json
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
)
from tqdm.auto import tqdm

from transformers import AutoModel
from peft import LoraConfig, get_peft_model

os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [ ]:
def _largest_component_mask(mask: np.ndarray) -> np.ndarray:
    """Mantém apenas o maior componente conectado da máscara binária."""
    mask = mask.astype("uint8")
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels <= 1:
        return mask.astype(bool)
    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return labels == largest


def _resize_for_detection(img: np.ndarray, max_side: int = 768):
    """Reduz a imagem para detectar o círculo mais rapidamente e retorna a escala."""
    H, W = img.shape[:2]
    max_dim = max(H, W)

    if max_dim <= max_side:
        return img, 1.0

    scale = float(max_side) / float(max_dim)
    new_w = max(1, int(round(W * scale)))
    new_h = max(1, int(round(H * scale)))
    small = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return small, scale


def detect_fundus_square(
    pil_img: Image.Image,
    padding: float = 1.03,
    return_mask: bool = False,
    detect_max_side: int = None,
):
    """Detecta a região retinal e retorna crop quadrado baseado no círculo envolvente.

    Versão otimizada:
    - a detecção do círculo é feita em uma cópia reduzida da imagem;
    - centro e raio são reescalados para a resolução original;
    - evita passar milhões de pixels para `cv2.minEnclosingCircle`, que era o gargalo.
    """
    img = np.array(pil_img.convert("RGB"))
    H, W = img.shape[:2]

    if detect_max_side is None:
        detect_max_side = int(CIRCLE_DETECT_MAX_SIDE)

    if H < 10 or W < 10:
        info = {
            "status": "too_small",
            "cx": W / 2,
            "cy": H / 2,
            "radius": min(H, W) / 2,
            "side": min(H, W),
            "detect_scale": 1.0,
            "mask_area_frac": 0.0,
        }
        mask_full = np.zeros((H, W), dtype=bool)
        return (pil_img, info, mask_full) if return_mask else (pil_img, info)

    small, scale = _resize_for_detection(img, max_side=detect_max_side)
    hS, wS = small.shape[:2]

    hsv = cv2.cvtColor(small, cv2.COLOR_RGB2HSV)
    gray = cv2.cvtColor(small, cv2.COLOR_RGB2GRAY)

    s = hsv[..., 1]
    v = hsv[..., 2]

    v_thr = max(10, int(np.percentile(v, 1)) + 5)
    gray_thr = max(10, int(np.percentile(gray, 1)) + 5)

    mask_bright = (v > v_thr) | (gray > gray_thr)
    mask_color = s > 8
    mask = mask_bright & (mask_color | (gray > 25))

    mask = mask.astype("uint8") * 255

    # Kernels proporcionais ao tamanho reduzido.
    k_open = max(3, int(round(min(hS, wS) * 0.010)))
    k_close = max(9, int(round(min(hS, wS) * 0.035)))
    if k_open % 2 == 0:
        k_open += 1
    if k_close % 2 == 0:
        k_close += 1

    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_open, k_open))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_close, k_close))

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)

    mask_bool_small = _largest_component_mask(mask > 0)

    if mask_bool_small.sum() < 0.05 * hS * wS:
        gray_fallback = gray > 10
        gray_fallback = (
            cv2.morphologyEx(
                gray_fallback.astype("uint8") * 255,
                cv2.MORPH_CLOSE,
                kernel_close,
            )
            > 0
        )
        mask_bool_small = _largest_component_mask(gray_fallback)

    ys, xs = np.where(mask_bool_small)

    if len(xs) < 10:
        cx_small, cy_small = wS / 2.0, hS / 2.0
        radius_small = max(hS, wS) / 2.0
        status = "fallback_full_image"
    else:
        # Amostragem defensiva: se a máscara tiver muitos pixels, usa um subconjunto.
        # Para um círculo, isso é suficiente e evita travamentos em imagens muito grandes.
        if len(xs) > 50000:
            idx = np.linspace(0, len(xs) - 1, 50000).astype(int)
            xs_fit = xs[idx]
            ys_fit = ys[idx]
        else:
            xs_fit = xs
            ys_fit = ys

        pts = np.column_stack([xs_fit, ys_fit]).astype(np.float32)
        (cx_small, cy_small), radius_small = cv2.minEnclosingCircle(pts)
        status = "ok"

    # Reescala centro/raio para a imagem original.
    inv_scale = 1.0 / float(scale)
    cx = float(cx_small * inv_scale)
    cy = float(cy_small * inv_scale)
    radius = float(radius_small * inv_scale)

    side = int(np.ceil(2.0 * radius * float(padding)))
    side = max(16, side)

    # Limite defensivo: não deixa o crop ficar absurdamente maior do que a imagem.
    max_reasonable_side = int(np.ceil(1.25 * max(H, W)))
    if side > max_reasonable_side:
        side = max_reasonable_side
        status = status + "_side_clipped"

    x0 = int(round(cx - side / 2))
    y0 = int(round(cy - side / 2))
    x1 = x0 + side
    y1 = y0 + side

    src_x0 = max(x0, 0)
    src_y0 = max(y0, 0)
    src_x1 = min(x1, W)
    src_y1 = min(y1, H)

    crop = np.zeros((side, side, 3), dtype=np.uint8)

    dst_x0 = src_x0 - x0
    dst_y0 = src_y0 - y0
    dst_x1 = dst_x0 + (src_x1 - src_x0)
    dst_y1 = dst_y0 + (src_y1 - src_y0)

    if src_x1 > src_x0 and src_y1 > src_y0:
        crop[dst_y0:dst_y1, dst_x0:dst_x1] = img[src_y0:src_y1, src_x0:src_x1]

    info = {
        "status": status,
        "cx": float(cx),
        "cy": float(cy),
        "radius": float(radius),
        "side": int(side),
        "x0": int(x0),
        "y0": int(y0),
        "x1": int(x1),
        "y1": int(y1),
        "mask_area_frac": float(mask_bool_small.mean()),
        "detect_scale": float(scale),
        "detect_shape": (int(hS), int(wS)),
    }

    crop_img = Image.fromarray(crop)

    if return_mask:
        # Para preview, devolve a máscara reduzida mesmo. Ela é suficiente para inspeção.
        return crop_img, info, mask_bool_small

    return crop_img, info


def pre_resize_before_crop(pil_img: Image.Image, max_side: int = None) -> Image.Image:
    """Reduz a imagem ainda no carregamento antes do crop/CLAHE.

    Como a entrada final do DINOv3 será 512×512, não precisamos manter imagens
    enormes no pipeline de CPU. Este passo é o que deixa o Dataset viável para treino.
    """
    if not PRE_RESIZE_BEFORE_CROP:
        return pil_img
    if max_side is None:
        max_side = int(PRE_RESIZE_MAX_SIDE)

    img = pil_img.copy()
    img.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
    return img


def robust_circle_crop(pil_img: Image.Image) -> Image.Image:
    if not USE_ROBUST_CIRCLE_CROP:
        return pil_img
    crop_img, _ = detect_fundus_square(
        pil_img,
        padding=CIRCLE_CROP_PADDING,
        return_mask=False,
        detect_max_side=CIRCLE_DETECT_MAX_SIDE,
    )
    return crop_img


def _truthy_from_any(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return 1.0 if float(x) > 0 else 0.0
    s = str(x).strip().lower()
    if s in {"1", "true", "yes", "y", "sim", "male", "m", "positive", "positivo"}:
        return 1.0
    if s in {"0", "false", "no", "n", "não", "nao", "negative", "negativo"}:
        return 0.0
    return np.nan


def infer_diabetes_series(df: pd.DataFrame) -> pd.Series:
    if "diabetes" in df.columns:
        return df["diabetes"].apply(_truthy_from_any).astype("float32")
    if "comorbidities" in df.columns:
        has_info = df["comorbidities"].notna() & df["comorbidities"].astype(
            str
        ).str.strip().ne("")
        has_dm = df["comorbidities"].fillna("").astype(str).str.lower().str.contains("diabet")
        out = pd.Series(np.nan, index=df.index, dtype="float32")
        out.loc[has_info] = 0.0
        out.loc[has_dm] = 1.0
        return out
    return pd.Series(np.nan, index=df.index, dtype="float32")


def infer_hypertension_series(df: pd.DataFrame) -> pd.Series:
    for col in ["hypertension", "systemic_hypertension"]:
        if col in df.columns:
            return df[col].apply(_truthy_from_any).astype("float32")
    if "comorbidities" in df.columns:
        has_info = df["comorbidities"].notna() & df["comorbidities"].astype(
            str
        ).str.strip().ne("")
        has_ht = (
            df["comorbidities"]
            .fillna("")
            .astype(str)
            .str.lower()
            .str.contains("hypertens|hipertens")
        )
        out = pd.Series(np.nan, index=df.index, dtype="float32")
        out.loc[has_info] = 0.0
        out.loc[has_ht] = 1.0
        return out
    return pd.Series(np.nan, index=df.index, dtype="float32")


def infer_sex_flags(df: pd.DataFrame):
    if "patient_sex" not in df.columns:
        z = pd.Series(np.nan, index=df.index, dtype="float32")
        return z.copy(), z.copy()

    s = df["patient_sex"].fillna("").astype(str).str.strip().str.lower()
    male = pd.Series(np.nan, index=df.index, dtype="float32")
    female = pd.Series(np.nan, index=df.index, dtype="float32")
    male_mask = s.isin(["1", "m", "male", "masculino", "homem"])
    female_mask = s.isin(["2", "f", "female", "feminino", "mulher"])
    male.loc[male_mask] = 1.0
    male.loc[female_mask] = 0.0
    female.loc[female_mask] = 1.0
    female.loc[male_mask] = 0.0
    return male, female


def add_fold_metadata_patch_features(
    train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame
):
    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()

    age_col = "patient_age" if "patient_age" in train_df.columns else None
    if age_col is None:
        age_min, age_max = 0.0, 100.0
    else:
        train_age = pd.to_numeric(train_df[age_col], errors="coerce")
        if train_age.notna().any():
            age_min = float(train_age.min())
            age_max = float(train_age.max())
            if age_max <= age_min:
                age_max = age_min + 1.0
        else:
            age_min, age_max = 0.0, 100.0

    def _apply(d: pd.DataFrame) -> pd.DataFrame:
        d = d.copy()
        age = (
            pd.to_numeric(d[age_col], errors="coerce")
            if age_col is not None
            else pd.Series(np.nan, index=d.index)
        )
        d["meta_age_present"] = age.notna().astype("float32")
        age_filled = age.fillna(age_min)
        d["meta_age_norm01"] = (
            ((age_filled - age_min) / (age_max - age_min)).clip(0, 1).astype("float32")
        )
        male, female = infer_sex_flags(d)
        d["meta_is_male"] = male.astype("float32")
        d["meta_is_female"] = female.astype("float32")
        d["meta_diabetes_yes"] = infer_diabetes_series(d).astype("float32")
        d["meta_hypertension_yes"] = infer_hypertension_series(d).astype("float32")
        return d

    params = {"age_min_train": age_min, "age_max_train": age_max, "age_col": age_col}
    return _apply(train_df), _apply(val_df), _apply(test_df), params


def insert_metadata_patch(
    img: torch.Tensor, row: pd.Series, patch_size: int = 32, quad: int = 16
) -> torch.Tensor:
    """Insere o patch 32×32 no canto superior esquerdo antes da normalização."""
    out = img.clone()
    H, W = out.shape[-2:]
    ps = min(patch_size, H, W)
    q = min(quad, ps // 2)
    out[:, :ps, :ps] = 0.0

    age_present = float(row.get("meta_age_present", 0.0))
    age_norm = float(row.get("meta_age_norm01", 0.0))
    is_male = row.get("meta_is_male", np.nan)
    is_female = row.get("meta_is_female", np.nan)
    diab = row.get("meta_diabetes_yes", np.nan)
    hyper = row.get("meta_hypertension_yes", np.nan)

    # superior esquerdo: idade / presença
    out[0, 0:q, 0:q] = age_norm
    out[1, 0:q, 0:q] = 0.75 if age_present >= 0.5 else 0.25
    out[2, 0:q, 0:q] = 0.0

    # superior direito: sexo
    x0 = ps - q
    out[0, 0:q, x0:ps] = 0.0
    if pd.isna(is_male) and pd.isna(is_female):
        out[1, 0:q, x0:ps] = 0.0
        out[2, 0:q, x0:ps] = 0.0
    elif not pd.isna(is_male) and float(is_male) >= 0.5:
        out[1, 0:q, x0:ps] = 0.0
        out[2, 0:q, x0:ps] = 0.5
    else:
        out[1, 0:q, x0:ps] = 0.5
        out[2, 0:q, x0:ps] = 0.5

    # inferior esquerdo: diabetes
    y0 = ps - q
    out[0, y0:ps, 0:q] = 0.25
    out[1, y0:ps, 0:q] = 0.0
    if pd.isna(diab):
        out[2, y0:ps, 0:q] = 0.0
    else:
        out[2, y0:ps, 0:q] = 0.75 if float(diab) >= 0.5 else 0.0

    # inferior direito: hipertensão
    out[0, y0:ps, x0:ps] = 0.0
    out[1, y0:ps, x0:ps] = 0.25
    if pd.isna(hyper):
        out[2, y0:ps, x0:ps] = 0.0
    else:
        out[2, y0:ps, x0:ps] = 0.75 if float(hyper) >= 0.5 else 0.0

    return out.clamp(0.0, 1.0)


In [ ]:
# ==========================================================
# Representações tabulares usadas por learned tokens e late fusion
# ==========================================================


def binary_to_category_code(values: pd.Series) -> pd.Series:
    """0=desconhecido, 1=não, 2=sim."""
    numeric = pd.to_numeric(values, errors="coerce")
    code = pd.Series(0, index=values.index, dtype="int64")
    code.loc[numeric.notna() & (numeric < 0.5)] = 1
    code.loc[numeric.notna() & (numeric >= 0.5)] = 2
    return code


def prepare_fold_metadata(train_df, val_df, test_df):
    """Calcula normalização e códigos usando somente o treino do fold."""
    train_df, val_df, test_df, params = add_fold_metadata_patch_features(
        train_df, val_df, test_df
    )

    def _add_codes(d):
        d = d.copy()
        d["meta_age_available"] = d["meta_age_present"].fillna(0).astype("int64")
        d["meta_age_norm"] = d["meta_age_norm01"].fillna(0).astype("float32")

        sex_code = pd.Series(0, index=d.index, dtype="int64")
        sex_code.loc[d["meta_is_male"].fillna(0) >= 0.5] = 1
        sex_code.loc[d["meta_is_female"].fillna(0) >= 0.5] = 2
        d["meta_sex_code"] = sex_code

        d["meta_diabetes_code"] = binary_to_category_code(d["meta_diabetes_yes"])
        d["meta_hypertension_code"] = binary_to_category_code(d["meta_hypertension_yes"])
        return d

    params["categorical_codes"] = {
        "sex": {"unknown": 0, "male": 1, "female": 2},
        "diabetes": {"unknown": 0, "no": 1, "yes": 2},
        "hypertension": {"unknown": 0, "no": 1, "yes": 2},
    }
    return _add_codes(train_df), _add_codes(val_df), _add_codes(test_df), params


def resolve_image_path(images_dir, image_id, default_ext=".jpg"):
    image_id = str(image_id)
    direct = os.path.join(images_dir, image_id)
    if os.path.isfile(direct):
        return direct
    root, ext = os.path.splitext(image_id)
    if ext:
        raise FileNotFoundError(direct)
    candidates = [
        os.path.join(images_dir, image_id + default_ext),
        os.path.join(images_dir, image_id + ".jpeg"),
        os.path.join(images_dir, image_id + ".png"),
    ]
    for path in candidates:
        if os.path.isfile(path):
            return path
    raise FileNotFoundError(f"Imagem não encontrada: {image_id}. Candidatos: {candidates}")


IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3, 1, 1)


def build_pre_normalization_transform(image_size=512, train=True):
    """Mesmo pipeline de transforms do experimento isolado wev_exp_7."""
    if train:
        return transforms.Compose(
            [
                transforms.Lambda(robust_circle_crop),
                transforms.Resize((image_size, image_size)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(degrees=15),
                transforms.ToTensor(),
            ]
        )

    return transforms.Compose(
        [
            transforms.Lambda(robust_circle_crop),
            transforms.Resize(
                (image_size, image_size),
                antialias=True,
            ),
            transforms.ToTensor(),
        ]
    )


class BRSETHRFusionDataset(Dataset):
    def __init__(self, df, images_dir, fusion_mode, transform=None, image_ext=".jpg"):
        if fusion_mode not in FUSION_MODES:
            raise ValueError(f"fusion_mode inválido: {fusion_mode}")
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.fusion_mode = fusion_mode
        self.transform = transform
        self.image_ext = image_ext

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = resolve_image_path(self.images_dir, row["image_id"], self.image_ext)
        with Image.open(path) as pil_img:
            img = pil_img.convert("RGB")
            img = pre_resize_before_crop(img, max_side=PRE_RESIZE_MAX_SIDE)
            if self.transform is not None:
                img = self.transform(img)

        if self.fusion_mode == "visual_patch":
            img = insert_metadata_patch(
                img, row, patch_size=META_PATCH_SIZE, quad=META_PATCH_QUAD
            )
        img = (img - IMAGENET_MEAN) / IMAGENET_STD

        metadata = {
            "age_norm": torch.tensor(float(row["meta_age_norm"]), dtype=torch.float32),
            "age_available": torch.tensor(int(row["meta_age_available"]), dtype=torch.long),
            "sex_code": torch.tensor(int(row["meta_sex_code"]), dtype=torch.long),
            "diabetes_code": torch.tensor(int(row["meta_diabetes_code"]), dtype=torch.long),
            "hypertension_code": torch.tensor(
                int(row["meta_hypertension_code"]), dtype=torch.long
            ),
        }
        label = torch.tensor(int(row["label"]), dtype=torch.long)
        return img, metadata, label


In [ ]:
class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha: torch.Tensor, gamma: float = 2.0):
        super().__init__()
        self.register_buffer("alpha", alpha)
        self.gamma = gamma

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=-1)
        probs = log_probs.exp()
        targets_oh = F.one_hot(targets, num_classes=logits.size(-1)).float()
        focal = (1.0 - probs) ** self.gamma
        ce = -targets_oh * log_probs
        return (self.alpha.unsqueeze(0) * focal * ce).sum(dim=-1).mean()


def compute_class_weights(labels: np.ndarray, beta: float = 0.999) -> torch.Tensor:
    """
    Effective-number class weighting (Cui et al.). É menos extremo que 1/frequência,
    o que ajuda quando também usamos oversampling.
    """
    counts = np.bincount(labels.astype(int), minlength=2).astype(float)
    counts = np.maximum(counts, 1.0)
    effective_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / effective_num
    weights = weights / weights.mean()  # média 1 para manter a escala da loss estável
    return torch.tensor(weights, dtype=torch.float32)


@dataclass
class BinaryMetrics:
    auc_roc: float
    auc_pr: float
    f1: float
    precision: float
    recall: float
    specificity: float
    balanced_accuracy: float
    accuracy: float
    threshold: float
    tn: int
    fp: int
    fn: int
    tp: int
    f1_class_0: float
    f1_class_1: float
    precision_class_0: float
    precision_class_1: float
    recall_class_0: float
    recall_class_1: float
    macro_f1: float
    weighted_f1: float

    def to_dict(self):
        return {
            "auc_roc": self.auc_roc,
            "auc_pr": self.auc_pr,
            "f1": self.f1,
            "precision": self.precision,
            "recall_sensitivity": self.recall,
            "specificity": self.specificity,
            "balanced_accuracy": self.balanced_accuracy,
            "accuracy": self.accuracy,
            "threshold": self.threshold,
            "tn": self.tn,
            "fp": self.fp,
            "fn": self.fn,
            "tp": self.tp,
            "f1_class_0": self.f1_class_0,
            "f1_class_1": self.f1_class_1,
            "precision_class_0": self.precision_class_0,
            "precision_class_1": self.precision_class_1,
            "recall_class_0": self.recall_class_0,
            "recall_class_1": self.recall_class_1,
            "macro_f1": self.macro_f1,
            "weighted_f1": self.weighted_f1,
        }


def best_f1_threshold(y_true, y_score):
    """Escolhe threshold na validação maximizando F1. Nunca use no teste."""
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    # quantis evitam testar milhares de thresholds idênticos em datasets grandes
    candidates = np.unique(np.quantile(y_score, np.linspace(0.01, 0.99, 199)))
    if len(candidates) == 0:
        return 0.5
    f1s = [f1_score(y_true, (y_score >= t).astype(int), zero_division=0) for t in candidates]
    return float(candidates[int(np.argmax(f1s))])


def compute_binary_metrics(y_true, y_score, threshold=None):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    if threshold is None:
        threshold = best_f1_threshold(y_true, y_score)
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    spec = tn / max(tn + fp, 1)

    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=[NEG_LABEL_NAME, POS_LABEL_NAME],
        output_dict=True,
        zero_division=0,
    )

    return BinaryMetrics(
        auc_roc=float(roc_auc_score(y_true, y_score)),
        auc_pr=float(average_precision_score(y_true, y_score)),
        f1=float(f1_score(y_true, y_pred, zero_division=0)),  # F1 da classe positiva
        precision=float(precision_score(y_true, y_pred, zero_division=0)),
        recall=float(recall_score(y_true, y_pred, zero_division=0)),
        specificity=float(spec),
        balanced_accuracy=float(balanced_accuracy_score(y_true, y_pred)),
        accuracy=float(accuracy_score(y_true, y_pred)),
        threshold=float(threshold),
        tn=int(tn),
        fp=int(fp),
        fn=int(fn),
        tp=int(tp),
        f1_class_0=float(report[NEG_LABEL_NAME]["f1-score"]),
        f1_class_1=float(report[POS_LABEL_NAME]["f1-score"]),
        precision_class_0=float(report[NEG_LABEL_NAME]["precision"]),
        precision_class_1=float(report[POS_LABEL_NAME]["precision"]),
        recall_class_0=float(report[NEG_LABEL_NAME]["recall"]),
        recall_class_1=float(report[POS_LABEL_NAME]["recall"]),
        macro_f1=float(report["macro avg"]["f1-score"]),
        weighted_f1=float(report["weighted avg"]["f1-score"]),
    )


def bootstrap_ci(y_true, y_score, metric_fn, n_boot=1000, seed=42, alpha=0.05):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    n = len(y_true)
    stats = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        # pula amostras bootstrap sem uma das classes
        if len(np.unique(y_true[idx])) < 2:
            continue
        try:
            stats.append(float(metric_fn(y_true[idx], y_score[idx])))
        except ValueError:
            continue
    if not stats:
        return float("nan"), float("nan"), float("nan")
    arr = np.array(stats)
    return (
        float(arr.mean()),
        float(np.quantile(arr, alpha / 2)),
        float(np.quantile(arr, 1 - alpha / 2)),
    )


In [ ]:
class LearnedMetadataTokenizer(nn.Module):
    """Produz quatro tokens com a mesma dimensão dos tokens do DINOv3."""

    def __init__(self, hidden_size, dropout=0.1):
        super().__init__()
        age_hidden = max(32, hidden_size // 4)
        self.age_encoder = nn.Sequential(
            nn.Linear(1, age_hidden), nn.GELU(), nn.Linear(age_hidden, hidden_size)
        )
        self.age_available_embedding = nn.Embedding(2, hidden_size)
        self.sex_embedding = nn.Embedding(3, hidden_size)
        self.diabetes_embedding = nn.Embedding(3, hidden_size)
        self.hypertension_embedding = nn.Embedding(3, hidden_size)
        self.token_type_embedding = nn.Parameter(torch.zeros(1, 4, hidden_size))
        nn.init.trunc_normal_(self.token_type_embedding, std=0.02)
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, metadata):
        age = metadata["age_norm"].float().unsqueeze(-1)
        age_token = self.age_encoder(age)
        age_token = age_token + self.age_available_embedding(metadata["age_available"].long())
        tokens = torch.stack(
            [
                age_token,
                self.sex_embedding(metadata["sex_code"].long()),
                self.diabetes_embedding(metadata["diabetes_code"].long()),
                self.hypertension_embedding(metadata["hypertension_code"].long()),
            ],
            dim=1,
        )
        return self.dropout(self.norm(tokens + self.token_type_embedding))


class LateFusionMetadataEncoder(nn.Module):
    def __init__(self, output_dim=128, dropout=0.1):
        super().__init__()
        emb_dim = 16
        self.sex_embedding = nn.Embedding(3, emb_dim)
        self.diabetes_embedding = nn.Embedding(3, emb_dim)
        self.hypertension_embedding = nn.Embedding(3, emb_dim)
        self.encoder = nn.Sequential(
            nn.Linear(2 + 3 * emb_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, output_dim),
            nn.LayerNorm(output_dim),
        )

    def forward(self, metadata):
        age = torch.stack(
            [
                metadata["age_norm"].float(),
                metadata["age_available"].float(),
            ],
            dim=-1,
        )
        x = torch.cat(
            [
                age,
                self.sex_embedding(metadata["sex_code"].long()),
                self.diabetes_embedding(metadata["diabetes_code"].long()),
                self.hypertension_embedding(metadata["hypertension_code"].long()),
            ],
            dim=-1,
        )
        return self.encoder(x)


class DINOv3FusionClassifier(nn.Module):
    def __init__(
        self,
        model_name,
        fusion_mode,
        num_classes=2,
        lora_r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=None,
        head_dropout=0.1,
        late_metadata_dim=128,
        use_gradient_checkpointing=False,
    ):
        super().__init__()
        if fusion_mode not in FUSION_MODES:
            raise ValueError(f"fusion_mode inválido: {fusion_mode}")
        self.fusion_mode = fusion_mode

        backbone = AutoModel.from_pretrained(model_name)
        hidden_size = int(backbone.config.hidden_size)
        for p in backbone.parameters():
            p.requires_grad = False

        target_modules = target_modules or LORA_TARGET_MODULES
        lora_cfg = LoraConfig(
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            target_modules=target_modules,
            bias="none",
            task_type=None,
        )
        self.backbone = get_peft_model(backbone, lora_cfg)
        if use_gradient_checkpointing:
            self.backbone.gradient_checkpointing_enable()

        self.metadata_tokenizer = None
        self.late_metadata_encoder = None
        if fusion_mode == "learned_tokens":
            self.metadata_tokenizer = LearnedMetadataTokenizer(hidden_size, head_dropout)
            head_input = hidden_size
        elif fusion_mode == "late_fusion":
            self.late_metadata_encoder = LateFusionMetadataEncoder(
                late_metadata_dim, head_dropout
            )
            head_input = hidden_size + late_metadata_dim
        else:
            head_input = hidden_size

        head_hidden = max(128, hidden_size // 2)
        self.head = nn.Sequential(
            nn.LayerNorm(head_input),
            nn.Dropout(head_dropout),
            nn.Linear(head_input, head_hidden),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_hidden, num_classes),
        )

    def core_backbone(self):
        return (
            self.backbone.get_base_model()
            if hasattr(self.backbone, "get_base_model")
            else self.backbone
        )

    def encode_standard(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values)
        if getattr(outputs, "pooler_output", None) is not None:
            return outputs.pooler_output
        core = self.core_backbone()
        n_prefix = 1 + int(getattr(core.config, "num_register_tokens", 0))
        return outputs.last_hidden_state[:, n_prefix:, :].mean(dim=1)

    def encode_with_learned_tokens(self, pixel_values, metadata):
        core = self.core_backbone()
        pixel_values = pixel_values.to(core.embeddings.patch_embeddings.weight.dtype)
        hidden_states = core.embeddings(pixel_values)
        n_prefix = 1 + int(getattr(core.config, "num_register_tokens", 0))
        prefix = hidden_states[:, :n_prefix, :]
        image_patches = hidden_states[:, n_prefix:, :]
        meta_tokens = self.metadata_tokenizer(metadata).to(hidden_states.dtype)
        hidden_states = torch.cat([prefix, meta_tokens, image_patches], dim=1)

        # O RoPE continua sendo calculado somente para os patches espaciais.
        position_embeddings = core.rope_embeddings(pixel_values)
        encoded = core.model(
            hidden_states,
            position_embeddings=position_embeddings,
            return_dict=True,
        )
        sequence = core.norm(encoded.last_hidden_state)
        return sequence[:, 0, :]  # CLS

    def forward(self, pixel_values, metadata):
        if self.fusion_mode == "learned_tokens":
            features = self.encode_with_learned_tokens(pixel_values, metadata)
        else:
            image_features = self.encode_standard(pixel_values)
            if self.fusion_mode == "late_fusion":
                meta_features = self.late_metadata_encoder(metadata).to(image_features.dtype)
                features = torch.cat([image_features, meta_features], dim=-1)
            else:
                features = image_features
        return self.head(features)

    def parameter_counts(self):
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        return trainable, total


In [ ]:
def move_metadata_to_device(metadata, target_device):
    return {k: v.to(target_device, non_blocking=True) for k, v in metadata.items()}


def print_trainable_modules(model):
    trainable, total = model.parameter_counts()
    print(f"Treináveis: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)")
    groups = {"lora": 0, "metadata": 0, "head": 0, "other": 0}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "lora_" in name:
            groups["lora"] += p.numel()
        elif "metadata_tokenizer" in name or "late_metadata_encoder" in name:
            groups["metadata"] += p.numel()
        elif name.startswith("head."):
            groups["head"] += p.numel()
        else:
            groups["other"] += p.numel()
    print(groups)


def smoke_test(backbone_key="small"):
    images = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    metadata = {
        "age_norm": torch.tensor([0.3, 0.7], device=device),
        "age_available": torch.tensor([1, 1], device=device),
        "sex_code": torch.tensor([1, 2], device=device),
        "diabetes_code": torch.tensor([1, 2], device=device),
        "hypertension_code": torch.tensor([1, 2], device=device),
    }
    for mode in FUSION_MODES:
        print(f"\nSmoke test: {backbone_key}/{mode}")
        model = DINOv3FusionClassifier(
            BACKBONES[backbone_key],
            mode,
            lora_r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            target_modules=LORA_TARGET_MODULES,
        ).to(device)
        print_trainable_modules(model)
        with torch.no_grad():
            logits = model(images, metadata)
        print("Logits:", tuple(logits.shape))
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


if RUN_SMOKE_TEST:
    smoke_test("small")


In [ ]:
# Constrói o DataFrame binário (Normal vs alvo) a partir do CSV bruto do BRSET.
# Confirme os nomes das colunas com seu CSV; o paper/listagens públicas podem variar nos nomes.
ALL_PATHOLOGY_COLS = [
    "diabetic_retinopathy",
    "diabetic_macular_edema",  # nome alternativo em algumas versões
    "macular_edema",  # nome usado no labels_brset.csv
    "scar",
    "nevus",
    "amd",  # ou "age_related_macular_degeneration"
    "vascular_occlusion",
    "hypertensive_retinopathy",
    "drusens",
    "hemorrhage",
    "retinal_detachment",
    "myopic_fundus",
    "increased_cup_disc",
    "other",
]


def _to_binary_series(s: pd.Series) -> pd.Series:
    """Torna robusto a CSVs com 0/1, True/False, strings ou NaN."""
    if s.dtype == bool:
        return s.astype(int)
    return pd.to_numeric(s, errors="coerce").fillna(0).astype(int).clip(0, 1)


def filter_quality_brset(df: pd.DataFrame, exclude_inadequate: bool = True) -> pd.DataFrame:
    """Remove imagens problemáticas do BRSET antes de montar Normal vs alvo.

    No labels_brset.csv usado aqui, não há coluna chamada `inadequates`.
    A coluna equivalente é `quality`, com valores `Adequate`/`Inadequate`.
    Ela é derivada dos critérios `focus`, `Illuminaton`, `image_field` e `artifacts`:
    valor 2 em qualquer um desses critérios marca a imagem como inadequada.
    """
    if not exclude_inadequate:
        print("Filtro de qualidade desativado: usando imagens Adequate + Inadequate.")
        return df.copy()

    df = df.copy()
    n0 = len(df)

    if "quality" in df.columns:
        keep = df["quality"].astype(str).str.lower().eq("adequate")
        reason = "quality == 'Adequate'"
    else:
        quality_cols = [
            c
            for c in ["focus", "Illuminaton", "illumination", "image_field", "artifacts"]
            if c in df.columns
        ]
        if not quality_cols:
            print(
                "[aviso] Não encontrei coluna `quality` nem critérios de qualidade; nenhum filtro aplicado."
            )
            return df
        # No BRSET, 1 = adequado e 2 = inadequado; 0 aparece raramente em focus e não deve ser removido automaticamente.
        keep = ~(df[quality_cols].apply(pd.to_numeric, errors="coerce") == 2).any(axis=1)
        reason = f"nenhum critério inadequado em {quality_cols}"

    out = df[keep].reset_index(drop=True)
    print(
        f"Filtro de qualidade ({reason}): removidas {n0 - len(out)} de {n0} imagens ({(n0-len(out))/n0:.1%})."
    )
    return out


def build_binary_dataframe(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    df = df.copy()
    present = [c for c in ALL_PATHOLOGY_COLS if c in df.columns]
    missing = [c for c in ALL_PATHOLOGY_COLS if c not in df.columns]
    if missing:
        print(f"[aviso] Colunas patológicas ausentes no CSV: {missing}")
    if target_col not in df.columns:
        raise ValueError(f"Coluna alvo '{target_col}' não está no CSV.")

    # Garante labels binários mesmo se o CSV vier com strings/NaN.
    for c in present:
        df[c] = _to_binary_series(df[c])

    normal_mask = df[present].sum(axis=1) == 0
    pos_mask = df[target_col] == 1

    normal_df = df[normal_mask].assign(label=0)
    pos_df = df[pos_mask].assign(label=1)

    out = pd.concat([normal_df, pos_df], ignore_index=True)
    out = out.drop_duplicates(subset=["image_id"], keep="last").reset_index(drop=True)
    out["label"] = out["label"].astype(int)

    print(
        f"Normais: {(out['label']==0).sum()} | "
        f"Positivos: {(out['label']==1).sum()} | "
        f"Prevalência positiva: {out['label'].mean():.4f} | "
        f"Pacientes únicos: {out['patient_id'].nunique()}"
    )
    return out


In [ ]:
raw_df = pd.read_csv(CSV_PATH)
print("Colunas do CSV:", list(raw_df.columns))
print(f"Total de linhas: {len(raw_df)}")

# Diagnóstico rápido da qualidade no BRSET
if "quality" in raw_df.columns:
    print("\nDistribuição da coluna quality:")
    print(raw_df["quality"].value_counts(dropna=False).to_string())

for c in ["focus", "Illuminaton", "image_field", "artifacts"]:
    if c in raw_df.columns:
        print(f"\n{c}:")
        print(raw_df[c].value_counts(dropna=False).sort_index().to_string())

# Neste experimento não removemos imagens por qualidade.
# A função abaixo respeita EXCLUDE_INADEQUATE_QUALITY=False.
raw_df = filter_quality_brset(raw_df, exclude_inadequate=EXCLUDE_INADEQUATE_QUALITY)

df = build_binary_dataframe(raw_df, target_col=TARGET_COL)
df.head()


In [ ]:
# ============================================================
# Estatísticas de dados faltantes dos metadados
# Deve ser executada DEPOIS de:
# df = build_binary_dataframe(raw_df, target_col=TARGET_COL)
# ============================================================


def metadata_missingness_summary(df, task_name):
    d = df.copy()

    # --------------------------------------------------------
    # Idade
    # --------------------------------------------------------
    if "patient_age" in d.columns:
        age = pd.to_numeric(
            d["patient_age"],
            errors="coerce",
        )
    else:
        age = pd.Series(
            np.nan,
            index=d.index,
        )

    # --------------------------------------------------------
    # Sexo
    # Usa exatamente a mesma função do pipeline final.
    # Um sexo é conhecido quando foi reconhecido como
    # masculino ou feminino.
    # --------------------------------------------------------
    male, female = infer_sex_flags(d)

    sex_known = male.notna() | female.notna()

    # --------------------------------------------------------
    # Diabetes e hipertensão
    # Usa exatamente as funções do pipeline final.
    # --------------------------------------------------------
    diabetes = infer_diabetes_series(d)

    hypertension = infer_hypertension_series(d)

    variables = {
        "Age": age.notna(),
        "Sex": sex_known,
        "Diabetes": diabetes.notna(),
        "Hypertension": hypertension.notna(),
    }

    rows = []

    for variable, available in variables.items():

        n_total = len(d)
        n_available = int(available.sum())
        n_missing = int((~available).sum())

        rows.append(
            {
                "Task": task_name,
                "Variable": variable,
                "N": n_total,
                "Available_n": n_available,
                "Available_pct": 100 * n_available / n_total,
                "Missing_n": n_missing,
                "Missing_pct": 100 * n_missing / n_total,
            }
        )

    result = pd.DataFrame(rows)

    print(f"\n{'=' * 70}")
    print(f"METADATA MISSINGNESS — {task_name}")
    print(f"{'=' * 70}")

    display(
        result.style.format(
            {
                "Available_pct": "{:.2f}",
                "Missing_pct": "{:.2f}",
            }
        )
    )

    # --------------------------------------------------------
    # Quantos registros possuem os quatro metadados completos?
    # --------------------------------------------------------
    complete = age.notna() & sex_known & diabetes.notna() & hypertension.notna()

    print(
        f"\nComplete metadata: "
        f"{complete.sum()} / {len(d)} "
        f"({100 * complete.mean():.2f}%)"
    )

    print(
        f"At least one missing metadata variable: "
        f"{(~complete).sum()} / {len(d)} "
        f"({100 * (~complete).mean():.2f}%)"
    )

    # --------------------------------------------------------
    # Opcional: estatísticas em nível de paciente
    # --------------------------------------------------------
    patient_result = None

    if "patient_id" in d.columns:

        patient_df = d.copy()

        patient_df["_age_available"] = age.notna().values
        patient_df["_sex_available"] = sex_known.values
        patient_df["_diabetes_available"] = diabetes.notna().values
        patient_df["_hypertension_available"] = hypertension.notna().values

        # Os metadados são patient-level, portanto uma linha por paciente.
        patient_df = patient_df.drop_duplicates(
            subset="patient_id",
            keep="first",
        )

        patient_variables = {
            "Age": patient_df["_age_available"],
            "Sex": patient_df["_sex_available"],
            "Diabetes": patient_df["_diabetes_available"],
            "Hypertension": patient_df["_hypertension_available"],
        }

        patient_rows = []

        for variable, available in patient_variables.items():

            n_total = len(patient_df)
            n_available = int(available.sum())
            n_missing = int((~available).sum())

            patient_rows.append(
                {
                    "Task": task_name,
                    "Variable": variable,
                    "Patients_n": n_total,
                    "Available_n": n_available,
                    "Available_pct": 100 * n_available / n_total,
                    "Missing_n": n_missing,
                    "Missing_pct": 100 * n_missing / n_total,
                }
            )

        patient_result = pd.DataFrame(patient_rows)

        print("\nPatient-level missingness:")

        display(
            patient_result.style.format(
                {
                    "Available_pct": "{:.2f}",
                    "Missing_pct": "{:.2f}",
                }
            )
        )

    return result, patient_result


# O TARGET_COL já está definido no notebook.
TASK_NAME = (
    "Diabetic retinopathy"
    if TARGET_COL == "diabetic_retinopathy"
    else "Hypertensive retinopathy"
)

missingness_image, missingness_patient = metadata_missingness_summary(
    df,
    TASK_NAME,
)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time


def _load_brset_image_for_crop_preview(image_id):
    candidates = [
        os.path.join(IMAGES_DIR, f"{image_id}.jpg"),
        os.path.join(IMAGES_DIR, f"{image_id}.jpeg"),
        os.path.join(IMAGES_DIR, f"{image_id}.png"),
        os.path.join(IMAGES_DIR, str(image_id)),
    ]
    for p in candidates:
        if os.path.exists(p):
            return Image.open(p).convert("RGB")
    raise FileNotFoundError(f"Imagem não encontrada para image_id={image_id}")


def preview_robust_circle_crop(
    df_in,
    n_examples=6,
    seed=SEED,
):
    # Seleciona exemplos positivos e negativos.
    pos = df_in[df_in["label"] == 1]
    neg = df_in[df_in["label"] == 0]

    n_pos = min(max(1, n_examples // 2), len(pos))
    n_neg = min(max(1, n_examples - n_pos), len(neg))

    parts = []
    if n_pos > 0:
        parts.append(pos.sample(n=n_pos, random_state=seed))
    if n_neg > 0:
        parts.append(neg.sample(n=n_neg, random_state=seed + 1))

    if parts:
        sample_df = pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    else:
        sample_df = df_in.sample(n=min(n_examples, len(df_in)), random_state=seed).reset_index(
            drop=True
        )

    resize_only = transforms.Compose(
        [
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),
        ]
    )

    fig, axes = plt.subplots(
        len(sample_df),
        4,
        figsize=(18, 4.2 * len(sample_df)),
    )

    if len(sample_df) == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, (_, row) in enumerate(sample_df.iterrows()):
        t0 = time.time()

        image_id = row["image_id"]
        raw_img_full = _load_brset_image_for_crop_preview(image_id)

        # Usa o mesmo pré-resize aplicado no Dataset.
        raw_img = pre_resize_before_crop(raw_img_full, max_side=PRE_RESIZE_MAX_SIDE)
        raw_np = np.array(raw_img.convert("RGB"))

        crop_img, info, mask = detect_fundus_square(
            raw_img,
            padding=CIRCLE_CROP_PADDING,
            return_mask=True,
            detect_max_side=CIRCLE_DETECT_MAX_SIDE,
        )

        final_img = resize_only(crop_img)

        crop_np = np.array(crop_img) / 255.0
        final_np = np.array(final_img) / 255.0

        lab = POS_LABEL_NAME if int(row["label"]) == 1 else NEG_LABEL_NAME
        quality_text = row.get("quality", "NA")

        axes[i, 0].imshow(raw_np)
        axes[i, 0].set_title(
            f"Original pré-redimensionada\nimage_id={image_id} | {quality_text} | {lab}",
            fontsize=9,
        )
        axes[i, 0].axis("off")

        circ = patches.Circle(
            (info["cx"], info["cy"]),
            info["radius"],
            fill=False,
            linewidth=2,
            edgecolor="yellow",
        )
        axes[i, 0].add_patch(circ)

        axes[i, 1].imshow(mask, cmap="gray")
        axes[i, 1].set_title(
            f"Máscara detectada\narea={info['mask_area_frac']:.3f}, status={info['status']}",
            fontsize=9,
        )
        axes[i, 1].axis("off")

        axes[i, 2].imshow(crop_np)
        axes[i, 2].set_title(
            f"Crop quadrado pelo círculo\nside={info['side']}, r={info['radius']:.1f}",
            fontsize=9,
        )
        axes[i, 2].axis("off")

        axes[i, 3].imshow(final_np)
        axes[i, 3].set_title(f"Resultado final {IMAGE_SIZE}×{IMAGE_SIZE}", fontsize=9)
        axes[i, 3].axis("off")

        print(
            f"[crop preview] {image_id}: {time.time() - t0:.2f}s | "
            f"status={info['status']} | side={info['side']} | "
            f"detect_scale={info.get('detect_scale', 'NA')}"
        )

    plt.tight_layout()
    plt.show()


preview_robust_circle_crop(
    df,
    n_examples=6,
    seed=SEED,
)


In [ ]:
# Inspeção do patient_id (importante: BRSET tem ~2 imagens/paciente, OD e OS).
# Se aparecer paciente com labels conflitantes (1 olho saudável, outro doente),
# isso é normal — mantemos as duas imagens, mas o sanity check da CV garante
# que o paciente fica inteiro num único split.
n_pat = df["patient_id"].nunique()
n_img = len(df)
print(
    f"Total: {n_img} imagens de {n_pat} pacientes únicos "
    f"(média {n_img/n_pat:.2f} img/paciente)"
)

# Distribuição de imagens por paciente
img_per_pat = df.groupby("patient_id").size()
print("\nImagens por paciente:")
print(img_per_pat.value_counts().sort_index().to_string())

# Pacientes com labels conflitantes (ex.: olho normal + olho com patologia)
labels_per_pat = df.groupby("patient_id")["label"].nunique()
n_conflict = (labels_per_pat > 1).sum()
print(f"\nPacientes com labels heterogêneos (OD vs OS): {n_conflict}")
if n_conflict > 0:
    print("  → ok mantê-los; apenas garante-se que todas as imagens vão pro mesmo split.")

# Duplicatas de image_id (não deveria haver após build_binary_dataframe)
dups = df["image_id"].duplicated().sum()
assert dups == 0, f"Há {dups} image_id duplicados — investigue antes de prosseguir."
print("\n✓ Sem image_id duplicados.")


In [ ]:
import matplotlib.pyplot as plt


def _safe_float(x, default=np.nan):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def _safe_int_or_na(x):
    try:
        if pd.isna(x):
            return "NA"
        return int(float(x))
    except Exception:
        return "NA"


def _format_sex(row):
    male_val = _safe_float(row.get("meta_is_male", np.nan))
    female_val = _safe_float(row.get("meta_is_female", np.nan))
    if not pd.isna(male_val) and male_val >= 0.5:
        return "M"
    if not pd.isna(female_val) and female_val >= 0.5:
        return "F"
    return "NA"


def _load_brset_image(image_id):
    candidates = [
        os.path.join(IMAGES_DIR, f"{image_id}.jpg"),
        os.path.join(IMAGES_DIR, f"{image_id}.jpeg"),
        os.path.join(IMAGES_DIR, f"{image_id}.png"),
        os.path.join(IMAGES_DIR, str(image_id)),
    ]
    for p in candidates:
        if os.path.exists(p):
            return Image.open(p).convert("RGB")
    raise FileNotFoundError(f"Imagem não encontrada para image_id={image_id}")


def _build_preview_split(df_in: pd.DataFrame):
    patient_df_preview = (
        df_in.groupby("patient_id")["label"]
        .max()
        .reset_index()
        .rename(columns={"label": "patient_label"})
    )

    outer_preview = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    tv_pat_idx, te_pat_idx = next(
        outer_preview.split(patient_df_preview, patient_df_preview["patient_label"])
    )
    tv_pat = patient_df_preview.iloc[tv_pat_idx].reset_index(drop=True)

    inner_n_splits = max(2, int(round(1 / VAL_SIZE)))
    inner_preview = StratifiedKFold(n_splits=inner_n_splits, shuffle=True, random_state=SEED)
    tr_pat_idx, va_pat_idx = next(inner_preview.split(tv_pat, tv_pat["patient_label"]))

    tr_pat = tv_pat.iloc[tr_pat_idx]
    va_pat = tv_pat.iloc[va_pat_idx]
    te_pat = patient_df_preview.iloc[te_pat_idx]

    tr_df_preview = df_in[df_in["patient_id"].isin(tr_pat["patient_id"])].reset_index(
        drop=True
    )
    va_df_preview = df_in[df_in["patient_id"].isin(va_pat["patient_id"])].reset_index(
        drop=True
    )
    te_df_preview = df_in[df_in["patient_id"].isin(te_pat["patient_id"])].reset_index(
        drop=True
    )
    return tr_df_preview, va_df_preview, te_df_preview


def preview_metadata_patch(
    df_in: pd.DataFrame,
    n_examples: int = 6,
    random_state: int = 42,
):
    tr_df_preview, va_df_preview, te_df_preview = _build_preview_split(df_in)

    tr_df_preview, va_df_preview, te_df_preview, meta_params_preview = (
        add_fold_metadata_patch_features(
            tr_df_preview,
            va_df_preview,
            te_df_preview,
        )
    )

    pool = pd.concat([tr_df_preview, va_df_preview], ignore_index=True)

    pos = pool[pool["label"] == 1]
    neg = pool[pool["label"] == 0]

    n_pos = min(max(1, n_examples // 2), len(pos))
    n_neg = min(max(1, n_examples - n_pos), len(neg))

    samples = []
    if n_pos > 0:
        samples.append(pos.sample(n=n_pos, random_state=random_state))
    if n_neg > 0:
        samples.append(neg.sample(n=n_neg, random_state=random_state + 1))

    if len(samples) == 0:
        raise ValueError("Não há exemplos disponíveis para visualização.")

    ex_df = (
        pd.concat(samples, ignore_index=True)
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )

    # Preview sem normalização ImageNet para enxergar o patch corretamente.
    vis_tf = transforms.Compose(
        [
            transforms.Lambda(
                lambda im: pre_resize_before_crop(im, max_side=PRE_RESIZE_MAX_SIDE)
            ),
            transforms.Lambda(robust_circle_crop),
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),
            transforms.ToTensor(),
        ]
    )

    fig, axes = plt.subplots(len(ex_df), 3, figsize=(14, 4 * len(ex_df)))
    if len(ex_df) == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, (_, row) in enumerate(ex_df.iterrows()):
        raw_img = _load_brset_image(row["image_id"])

        base = vis_tf(raw_img)
        patched = insert_metadata_patch(
            base, row, patch_size=META_PATCH_SIZE, quad=META_PATCH_QUAD
        )

        base_np = base.permute(1, 2, 0).numpy().clip(0, 1)
        patched_np = patched.permute(1, 2, 0).numpy().clip(0, 1)
        zoom_np = patched_np[:112, :112, :]

        age_raw = row.get("patient_age", np.nan)
        age_present = _safe_int_or_na(row.get("meta_age_present", np.nan))
        age_norm = row.get("meta_age_norm01", np.nan)
        sex = _format_sex(row)
        diab = _safe_int_or_na(row.get("meta_diabetes_yes", np.nan))
        hyper = _safe_int_or_na(row.get("meta_hypertension_yes", np.nan))
        lab = POS_LABEL_NAME if int(row["label"]) == 1 else NEG_LABEL_NAME
        quality_text = row.get("quality", "NA")

        axes[i, 0].imshow(base_np)
        axes[i, 0].set_title(
            f"Sem patch\n{row['image_id']} | {quality_text} | {lab}", fontsize=9
        )
        axes[i, 0].axis("off")

        axes[i, 1].imshow(patched_np)
        axes[i, 1].add_patch(
            plt.Rectangle(
                (0, 0),
                META_PATCH_SIZE,
                META_PATCH_SIZE,
                fill=False,
                edgecolor="white",
                linewidth=2,
            )
        )
        axes[i, 1].set_title(f"Com patch {META_PATCH_SIZE}×{META_PATCH_SIZE}", fontsize=9)
        axes[i, 1].axis("off")

        axes[i, 2].imshow(zoom_np, interpolation="nearest")
        axes[i, 2].add_patch(
            plt.Rectangle(
                (0, 0),
                META_PATCH_SIZE,
                META_PATCH_SIZE,
                fill=False,
                edgecolor="white",
                linewidth=2,
            )
        )
        axes[i, 2].set_title("Zoom do canto superior esquerdo", fontsize=9)
        axes[i, 2].axis("off")

        axes[i, 0].text(
            0,
            -20,
            (
                f"idade={age_raw} | age_norm={age_norm:.3f} | presente={age_present} | "
                f"sexo={sex} | DM={diab} | HAS={hyper}"
            ),
            fontsize=9,
            va="top",
            transform=axes[i, 0].transData,
        )

    plt.tight_layout()
    plt.show()

    print("Parâmetros de normalização da idade usados na visualização:", meta_params_preview)
    print(
        f"Tamanho atual do patch: META_PATCH_SIZE={META_PATCH_SIZE}, META_PATCH_QUAD={META_PATCH_QUAD}"
    )


# Rode esta linha para verificar se o patch ficou visualmente adequado.
preview_metadata_patch(
    df,
    n_examples=6,
    random_state=SEED,
)


In [ ]:
def amp_dtype_for_device(device):
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


@torch.no_grad()
def evaluate(model, loader, device, use_amp=True):
    model.eval()
    all_scores, all_labels = [], []
    amp_dtype = amp_dtype_for_device(device)
    amp_enabled = bool(use_amp and device.type == "cuda")
    for images, metadata, labels in loader:
        images = images.to(device, non_blocking=True)
        metadata = move_metadata_to_device(metadata, device)
        with torch.amp.autocast(device_type=device.type, enabled=amp_enabled, dtype=amp_dtype):
            logits = model(images, metadata)
        probs = torch.softmax(logits, dim=-1)[:, 1].float().cpu().numpy()
        all_scores.append(probs)
        all_labels.append(labels.numpy())
    return np.concatenate(all_labels), np.concatenate(all_scores)


def seed_worker(worker_id):
    """Inicializa NumPy e random de cada worker a partir da seed do DataLoader."""
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_train_loader(dataset, train_df, batch_size, seed):
    """Loader determinístico e independente da ordem dos experimentos."""
    generator = torch.Generator()
    generator.manual_seed(seed)

    kwargs = dict(
        batch_size=batch_size,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
        worker_init_fn=seed_worker,
        generator=generator,
    )

    if USE_OVERSAMPLING:
        labels = train_df["label"].values.astype(int)
        counts = np.bincount(labels, minlength=2)
        sample_weights = (1.0 / np.maximum(counts, 1))[labels]
        sampler = WeightedRandomSampler(
            torch.as_tensor(sample_weights, dtype=torch.double),
            num_samples=len(sample_weights),
            replacement=True,
            generator=generator,
        )
        return DataLoader(
            dataset,
            sampler=sampler,
            **kwargs,
        )

    return DataLoader(
        dataset,
        shuffle=True,
        **kwargs,
    )


def make_eval_loader(dataset, batch_size):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )


def trainable_state_dict(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in names}


def build_optimizer(model):
    lora_params, fast_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (lora_params if "lora_" in name else fast_params).append(p)
    groups = []
    if lora_params:
        groups.append({"params": lora_params, "lr": LORA_LR})
    if fast_params:
        groups.append({"params": fast_params, "lr": HEAD_METADATA_LR})
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)


def train_one_fold(backbone_key, fusion_mode, fold_idx, train_df, val_df, test_df):
    # Garante que image_only, visual_patch, learned_tokens e late_fusion
    # iniciem com o mesmo estado aleatório no mesmo fold. Assim, a ordem
    # da grade experimental não altera a inicialização da cabeça/LoRA,
    # o embaralhamento, as augmentations nem o dropout.
    run_seed = SEED + int(fold_idx)
    if RESET_SEED_PER_FOLD_AND_STRATEGY:
        set_seed(run_seed)

    experiment = f"dinov3_{backbone_key}_{fusion_mode}"
    fold_dir = os.path.join(OUTPUT_DIR, experiment, f"fold_{fold_idx}")
    os.makedirs(fold_dir, exist_ok=True)
    summary_path = os.path.join(fold_dir, "test_summary.json")
    if SKIP_COMPLETED and os.path.isfile(summary_path):
        print(f"[skip] {experiment} fold {fold_idx}")
        with open(summary_path) as f:
            return json.load(f)

    train_df, val_df, test_df, meta_params = prepare_fold_metadata(train_df, val_df, test_df)
    with open(os.path.join(fold_dir, "metadata_parameters.json"), "w") as f:
        json.dump(meta_params, f, indent=2)

    effective_config = {
        "protocol_tag": PROTOCOL_TAG,
        "backbone_key": backbone_key,
        "model_name": BACKBONES[backbone_key],
        "fusion_mode": fusion_mode,
        "fold": int(fold_idx),
        "run_seed": int(run_seed),
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE_BY_BACKBONE[backbone_key],
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "effective_batch_size": (BATCH_SIZE_BY_BACKBONE[backbone_key] * GRAD_ACCUM_STEPS),
        "num_workers": NUM_WORKERS,
        "epochs": EPOCHS,
        "patience": PATIENCE,
        "lora_lr": LORA_LR,
        "head_metadata_lr": HEAD_METADATA_LR,
        "weight_decay": WEIGHT_DECAY,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "lora_target_modules": LORA_TARGET_MODULES,
        "focal_gamma": FOCAL_GAMMA,
        "class_weight_beta": CLASS_WEIGHT_BETA,
        "use_oversampling": USE_OVERSAMPLING,
        "use_clahe": USE_CLAHE,
        "circle_crop_padding": CIRCLE_CROP_PADDING,
        "circle_detect_max_side": CIRCLE_DETECT_MAX_SIDE,
        "pre_resize_max_side": PRE_RESIZE_MAX_SIDE,
    }
    with open(
        os.path.join(fold_dir, "effective_run_config.json"),
        "w",
    ) as f:
        json.dump(effective_config, f, indent=2)

    train_tf = build_pre_normalization_transform(IMAGE_SIZE, train=True)
    eval_tf = build_pre_normalization_transform(IMAGE_SIZE, train=False)
    train_ds = BRSETHRFusionDataset(train_df, IMAGES_DIR, fusion_mode, train_tf, IMAGE_EXT)
    val_ds = BRSETHRFusionDataset(val_df, IMAGES_DIR, fusion_mode, eval_tf, IMAGE_EXT)
    test_ds = BRSETHRFusionDataset(test_df, IMAGES_DIR, fusion_mode, eval_tf, IMAGE_EXT)
    batch_size = BATCH_SIZE_BY_BACKBONE[backbone_key]
    train_loader = make_train_loader(train_ds, train_df, batch_size, run_seed)
    val_loader = make_eval_loader(val_ds, batch_size)
    test_loader = make_eval_loader(test_ds, batch_size)

    model = DINOv3FusionClassifier(
        BACKBONES[backbone_key],
        fusion_mode,
        lora_r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        use_gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    ).to(device)
    print(f"\n{'='*70}\n{experiment} | fold {fold_idx}\n{'='*70}")
    print_trainable_modules(model)

    class_weights = compute_class_weights(train_df["label"].values, beta=CLASS_WEIGHT_BETA).to(
        device
    )
    criterion = BinaryFocalLoss(class_weights, gamma=FOCAL_GAMMA).to(device)
    optimizer = build_optimizer(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    amp_enabled = bool(USE_AMP and device.type == "cuda")
    amp_dtype = amp_dtype_for_device(device)
    scaler = torch.amp.GradScaler("cuda", enabled=(amp_enabled and amp_dtype == torch.float16))
    print(
        f"AMP={amp_enabled}, dtype={amp_dtype}, batch físico={batch_size}, "
        f"batch efetivo={batch_size * GRAD_ACCUM_STEPS}"
    )

    ckpt_path = os.path.join(fold_dir, "best_trainable_state.pt")
    history, best_auc_pr, best_threshold, pat = [], -np.inf, 0.5, 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        pbar = tqdm(
            train_loader,
            desc=f"{backbone_key}/{fusion_mode} f{fold_idx} ep {epoch}/{EPOCHS}",
        )
        for step, (images, metadata, labels) in enumerate(pbar, 1):
            images = images.to(device, non_blocking=True)
            metadata = move_metadata_to_device(metadata, device)
            labels = labels.to(device, non_blocking=True)
            with torch.amp.autocast(
                device_type=device.type, enabled=amp_enabled, dtype=amp_dtype
            ):
                logits = model(images, metadata)
                raw_loss = criterion(logits, labels)
                loss = raw_loss / max(GRAD_ACCUM_STEPS, 1)
            scaler.scale(loss).backward()
            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
                if MAX_GRAD_NORM and MAX_GRAD_NORM > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        [p for p in model.parameters() if p.requires_grad],
                        MAX_GRAD_NORM,
                    )
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            running_loss += raw_loss.item() * images.size(0)
            pbar.set_postfix(loss=f"{raw_loss.item():.4f}")

        scheduler.step()
        train_loss = running_loss / len(train_ds)
        y_val, s_val = evaluate(model, val_loader, device, USE_AMP)
        val_thr = best_f1_threshold(y_val, s_val)
        vm = compute_binary_metrics(y_val, s_val, threshold=val_thr)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                **{f"val_{k}": v for k, v in vm.to_dict().items()},
            }
        )
        print(
            f"[ep {epoch:02d}] loss={train_loss:.4f} | "
            f"AUC-PR={vm.auc_pr:.4f} | AUC={vm.auc_roc:.4f} | "
            f"F1={vm.f1:.4f} | sens={vm.recall:.4f} | "
            f"spec={vm.specificity:.4f} | thr={vm.threshold:.4f}"
        )

        if vm.auc_pr > best_auc_pr:
            best_auc_pr, best_threshold, pat = vm.auc_pr, vm.threshold, 0
            torch.save(
                {
                    "state_dict": trainable_state_dict(model),
                    "best_auc_pr": best_auc_pr,
                    "best_threshold": best_threshold,
                    "epoch": epoch,
                },
                ckpt_path,
            )
            print("  ✓ melhor checkpoint salvo")
        else:
            pat += 1
            print(f"  ✗ paciência {pat}/{PATIENCE}")
            if pat >= PATIENCE:
                print("Early stopping.")
                break

    pd.DataFrame(history).to_csv(os.path.join(fold_dir, "history.csv"), index=False)
    saved = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(saved["state_dict"], strict=False)
    best_threshold = float(saved["best_threshold"])

    y_test, s_test = evaluate(model, test_loader, device, USE_AMP)
    tm = compute_binary_metrics(y_test, s_test, threshold=best_threshold)
    _, roc_lo, roc_hi = bootstrap_ci(y_test, s_test, roc_auc_score, N_BOOT, SEED + fold_idx)
    _, pr_lo, pr_hi = bootstrap_ci(
        y_test, s_test, average_precision_score, N_BOOT, SEED + 100 + fold_idx
    )
    fixed_thr = best_threshold
    f1_fn = lambda yt, ys: f1_score(yt, (ys >= fixed_thr).astype(int), zero_division=0)
    _, f1_lo, f1_hi = bootstrap_ci(y_test, s_test, f1_fn, N_BOOT, SEED + 200 + fold_idx)

    summary = {
        "experiment": experiment,
        "backbone": backbone_key,
        "model_name": BACKBONES[backbone_key],
        "fusion_mode": fusion_mode,
        "fold": fold_idx,
        **tm.to_dict(),
        "auc_roc_ci_low": roc_lo,
        "auc_roc_ci_high": roc_hi,
        "auc_pr_ci_low": pr_lo,
        "auc_pr_ci_high": pr_hi,
        "f1_ci_low": f1_lo,
        "f1_ci_high": f1_hi,
        "best_epoch": int(saved["epoch"]),
        "best_val_auc_pr": float(saved["best_auc_pr"]),
    }
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)
    pd.DataFrame(
        {
            "image_id": test_df["image_id"].values,
            "patient_id": test_df["patient_id"].values,
            "label": y_test,
            "score": s_test,
            "threshold_from_val": best_threshold,
        }
    ).to_csv(os.path.join(fold_dir, "test_predictions.csv"), index=False)

    print(
        f"TESTE: AUC={tm.auc_roc:.4f}, AUC-PR={tm.auc_pr:.4f}, "
        f"F1={tm.f1:.4f}, sens={tm.recall:.4f}, "
        f"spec={tm.specificity:.4f}, balacc={tm.balanced_accuracy:.4f}"
    )
    del model, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary


In [ ]:
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)


def build_patient_folds(dataframe):
    patient_df = (
        dataframe.groupby("patient_id")["label"]
        .max()
        .reset_index()
        .rename(columns={"label": "patient_label"})
    )
    outer = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    folds, split_records = [], []
    for fold_idx, (tv_idx, te_idx) in enumerate(
        outer.split(patient_df, patient_df["patient_label"])
    ):
        tv_pat = patient_df.iloc[tv_idx].reset_index(drop=True)
        te_pat = patient_df.iloc[te_idx].reset_index(drop=True)
        inner = StratifiedKFold(
            n_splits=max(2, int(round(1 / VAL_SIZE))),
            shuffle=True,
            random_state=SEED + fold_idx,
        )
        tr_idx, va_idx = next(inner.split(tv_pat, tv_pat["patient_label"]))
        tr_pat, va_pat = tv_pat.iloc[tr_idx], tv_pat.iloc[va_idx]
        tr_df = dataframe[dataframe["patient_id"].isin(tr_pat["patient_id"])].reset_index(
            drop=True
        )
        va_df = dataframe[dataframe["patient_id"].isin(va_pat["patient_id"])].reset_index(
            drop=True
        )
        te_df = dataframe[dataframe["patient_id"].isin(te_pat["patient_id"])].reset_index(
            drop=True
        )

        tr_ids, va_ids, te_ids = (
            set(tr_df.patient_id),
            set(va_df.patient_id),
            set(te_df.patient_id),
        )
        assert not (tr_ids & va_ids or tr_ids & te_ids or va_ids & te_ids)
        print(
            f"Fold {fold_idx}: pacientes {len(tr_ids)}/{len(va_ids)}/{len(te_ids)} | "
            f"imagens {len(tr_df)}/{len(va_df)}/{len(te_df)} | "
            f"positivos {tr_df.label.sum()}/{va_df.label.sum()}/{te_df.label.sum()}"
        )
        folds.append((tr_df, va_df, te_df))
        for split_name, ids in [("train", tr_ids), ("val", va_ids), ("test", te_ids)]:
            split_records += [
                {"fold": fold_idx, "split": split_name, "patient_id": pid}
                for pid in sorted(ids)
            ]
    pd.DataFrame(split_records).to_csv(
        os.path.join(OUTPUT_DIR, "patient_wise_splits.csv"), index=False
    )
    return folds


PATIENT_FOLDS = build_patient_folds(df)
all_summaries = []
for backbone_key, fusion_mode in EXPERIMENTS_TO_RUN:
    for fold_idx in FOLDS_TO_RUN:
        tr_df, va_df, te_df = PATIENT_FOLDS[fold_idx]
        summary = train_one_fold(backbone_key, fusion_mode, fold_idx, tr_df, va_df, te_df)
        all_summaries.append(summary)
        pd.DataFrame(all_summaries).to_csv(
            os.path.join(OUTPUT_DIR, "all_fold_results_running.csv"), index=False
        )


In [ ]:
# Recupera todos os resultados existentes, inclusive os pulados por SKIP_COMPLETED.
existing = []
for backbone_key, fusion_mode in EXPERIMENTS_TO_RUN:
    experiment = f"dinov3_{backbone_key}_{fusion_mode}"
    for fold_idx in FOLDS_TO_RUN:
        path = os.path.join(OUTPUT_DIR, experiment, f"fold_{fold_idx}", "test_summary.json")
        if os.path.isfile(path):
            with open(path) as f:
                existing.append(json.load(f))

results_df = (
    pd.DataFrame(existing)
    .drop_duplicates(["backbone", "fusion_mode", "fold"], keep="last")
    .sort_values(["backbone", "fusion_mode", "fold"])
    .reset_index(drop=True)
)
results_df.to_csv(os.path.join(OUTPUT_DIR, "all_fold_results.csv"), index=False)

metrics = [
    "auc_roc",
    "auc_pr",
    "f1",
    "precision",
    "recall_sensitivity",
    "specificity",
    "balanced_accuracy",
    "accuracy",
    "macro_f1",
    "weighted_f1",
]
rows = []
for (backbone, fusion), group in results_df.groupby(["backbone", "fusion_mode"], sort=False):
    row = {"backbone": backbone, "fusion_mode": fusion, "n_folds": len(group)}
    for metric in metrics:
        row[f"{metric}_mean"] = float(group[metric].mean())
        row[f"{metric}_std"] = float(group[metric].std(ddof=0))
    rows.append(row)
cv_summary_df = pd.DataFrame(rows)
cv_summary_df.to_csv(os.path.join(OUTPUT_DIR, "cv_summary_by_strategy.csv"), index=False)


def ms(mean, std):
    return f"{mean:.4f} ± {std:.4f}"


formatted = []
for _, r in cv_summary_df.iterrows():
    formatted.append(
        {
            "Backbone": r.backbone,
            "Fusion": r.fusion_mode,
            "ROC-AUC": ms(r.auc_roc_mean, r.auc_roc_std),
            "AUC-PR": ms(r.auc_pr_mean, r.auc_pr_std),
            "F1": ms(r.f1_mean, r.f1_std),
            "Sensitivity": ms(r.recall_sensitivity_mean, r.recall_sensitivity_std),
            "Specificity": ms(r.specificity_mean, r.specificity_std),
            "Balanced accuracy": ms(r.balanced_accuracy_mean, r.balanced_accuracy_std),
        }
    )
formatted_df = pd.DataFrame(formatted)
formatted_df.to_csv(os.path.join(OUTPUT_DIR, "formatted_comparison_table.csv"), index=False)
display(results_df)
display(formatted_df)


In [ ]:
def plot_experiment_curves(backbone_key, fusion_mode):
    experiment = f"dinov3_{backbone_key}_{fusion_mode}"
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for fold_idx in FOLDS_TO_RUN:
        pred_path = os.path.join(
            OUTPUT_DIR, experiment, f"fold_{fold_idx}", "test_predictions.csv"
        )
        if not os.path.isfile(pred_path):
            continue
        pred = pd.read_csv(pred_path)
        fpr, tpr, _ = roc_curve(pred.label, pred.score)
        precision, recall, _ = precision_recall_curve(pred.label, pred.score)
        row = results_df[
            (results_df.backbone == backbone_key)
            & (results_df.fusion_mode == fusion_mode)
            & (results_df.fold == fold_idx)
        ].iloc[0]
        axes[0].plot(fpr, tpr, label=f"fold {fold_idx} (AUC={row.auc_roc:.3f})")
        axes[1].plot(recall, precision, label=f"fold {fold_idx} (AUC-PR={row.auc_pr:.3f})")
    axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
    axes[0].set(
        xlabel="False Positive Rate (1 - Specificity)",
        ylabel="True Positive Rate (Sensitivity)",
        title=f"ROC — {backbone_key}/{fusion_mode}",
    )
    axes[1].set(
        xlabel="Recall",
        ylabel="Precision",
        title=f"Precision–Recall — {backbone_key}/{fusion_mode}",
    )
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.savefig(
        os.path.join(OUTPUT_DIR, experiment, "roc_pr_curves.png"),
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()


for backbone_key, fusion_mode in EXPERIMENTS_TO_RUN:
    plot_experiment_curves(backbone_key, fusion_mode)


In [ ]:
from google.colab import runtime

runtime.unassign()
